# Notebook 00 — Setup and IFC Exploration

Verifies the environment, installs dependencies, and explores the IFC reference model schema.

**No GPU required.** Run on CPU runtime.

> **Always run Cell 1 first every session.**

In [ ]:
# ── Cell 1: Clone, install, set working directory ─────────────────────────────
import os, sys

REPO = 'ifc-graphrag-dt'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
    print('Repository updated.')
else:
    !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git
    print('Repository cloned.')

!bash {REPO}/colab_setup.sh

os.chdir(REPO)
REPO = '.'
print(f'Working directory: {os.getcwd()}')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print('Setup complete.')


In [ ]:
# ── Cell 2: Verify imports ────────────────────────────────────────────────────
import networkx as nx
import numpy as np
import json
from pathlib import Path

from benchmark.dtah_bench import DTAHBench
from evaluation.metrics.kcs_dt import KCSDTScorer
from evaluation.metrics.ggs import GGSScorer
from pipeline.layer1_retriever.khop_traversal import KHopTraversal

print('All core imports successful.')


In [ ]:
# ── Cell 3: DTAH-Bench overview ───────────────────────────────────────────────
bench = DTAHBench()
stats = bench.stats()
print('DTAH-Bench statistics:')
for k, v in stats.items():
    print(f'  {k}: {v}')

print('\nSample Tier 1 prompt:')
t1 = bench.load_tier(1)
print(json.dumps(t1[0], indent=2))


In [ ]:
# ── Cell 4: Prompt distribution chart ─────────────────────────────────────────
from collections import Counter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('outputs/figures', exist_ok=True)

all_prompts   = bench.load_all()
tier_counts   = Counter(p['tier'] for p in all_prompts)
domain_counts = Counter(p['domain'] for p in all_prompts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([f'Tier {t}' for t in sorted(tier_counts)],
            [tier_counts[t] for t in sorted(tier_counts)],
            color=['#2E5FA3','#1D9E75','#D85A30'])
axes[0].set_title('Prompts per Tier')
axes[0].set_ylabel('Count')
axes[1].bar(list(domain_counts.keys()), list(domain_counts.values()),
            color=['#2E5FA3','#1D9E75','#D85A30'])
axes[1].set_title('Prompts per Domain')
plt.tight_layout()
plt.savefig('outputs/figures/00_prompt_distribution.png', dpi=150)
plt.show()
print(f'Total prompts: {len(all_prompts)}')


In [ ]:
# ── Cell 5: Download IFC reference model ─────────────────────────────────────
import os, subprocess

IFC_DIR     = 'benchmark/ifc_reference_models'
DUPLEX_PATH = f'{IFC_DIR}/duplex.ifc'
os.makedirs(IFC_DIR, exist_ok=True)

# buildingSMART provides this file at multiple URLs.
# We try each in turn until we get a file > 100 KB.
URLS = [
    # Direct raw GitHub link
    'https://raw.githubusercontent.com/buildingSMART/Sample-Test-Files/master/IFC%202x3/Duplex%20Apartment/Duplex_A_20110907.ifc',
    # GitHub blob redirect
    'https://github.com/buildingSMART/Sample-Test-Files/raw/master/IFC%202x3/Duplex%20Apartment/Duplex_A_20110907.ifc',
    # IfcOpenShell test files mirror
    'https://github.com/IfcOpenShell/IfcOpenShell/raw/v0.7.0/test/files/IFC%202x3/Duplex_A_20110907.ifc',
]

def file_ok(path, min_bytes=100_000):
    return os.path.exists(path) and os.path.getsize(path) >= min_bytes

if file_ok(DUPLEX_PATH):
    print(f'Already downloaded: {DUPLEX_PATH} ({os.path.getsize(DUPLEX_PATH)/1024:.0f} KB)')
else:
    downloaded = False
    for i, url in enumerate(URLS, 1):
        print(f'Attempt {i}/{len(URLS)}: {url[:70]}...')
        try:
            result = subprocess.run(
                ['wget', '-q', '--timeout=30', '--tries=2', '-O', DUPLEX_PATH, url],
                capture_output=True, text=True
            )
            if file_ok(DUPLEX_PATH):
                print(f'  Downloaded: {os.path.getsize(DUPLEX_PATH)/1024:.0f} KB')
                downloaded = True
                break
            else:
                sz = os.path.getsize(DUPLEX_PATH) if os.path.exists(DUPLEX_PATH) else 0
                print(f'  File too small ({sz} bytes), trying next URL...')
                os.remove(DUPLEX_PATH) if os.path.exists(DUPLEX_PATH) else None
        except Exception as e:
            print(f'  Error: {e}')

    if not downloaded:
        # Final fallback: use requests with a session
        print('wget failed, trying requests...')
        import requests
        for url in URLS:
            try:
                r = requests.get(url, timeout=60, headers={'User-Agent': 'Mozilla/5.0'})
                if r.status_code == 200 and len(r.content) > 100_000:
                    with open(DUPLEX_PATH, 'wb') as f:
                        f.write(r.content)
                    print(f'Downloaded via requests: {len(r.content)/1024:.0f} KB')
                    downloaded = True
                    break
                else:
                    print(f'  Status {r.status_code}, size {len(r.content)} bytes')
            except Exception as e:
                print(f'  requests error: {e}')

    if not downloaded:
        raise RuntimeError(
            'All download attempts failed.\n'
            'Manual fix: download duplex.ifc from\n'
            'https://github.com/buildingSMART/Sample-Test-Files\n'
            'and upload it to benchmark/ifc_reference_models/duplex.ifc'
        )

print(f'\nFile ready: {os.path.abspath(DUPLEX_PATH)} ({os.path.getsize(DUPLEX_PATH)/1024:.0f} KB)')


In [ ]:
# ── Cell 6: Explore IFC schema ────────────────────────────────────────────────
import ifcopenshell

print(f'Opening: {os.path.abspath(DUPLEX_PATH)}')
model = ifcopenshell.open(DUPLEX_PATH)
print(f'IFC schema: {model.schema}')

type_counts = {}
for entity in model:
    t = entity.is_a()
    type_counts[t] = type_counts.get(t, 0) + 1

sorted_types = sorted(type_counts.items(), key=lambda x: -x[1])
print(f'\nTotal entities:      {sum(type_counts.values())}')
print(f'Unique entity types: {len(type_counts)}')
print('\nTop 20 entity types:')
for t, n in sorted_types[:20]:
    print(f'  {t:<45} {n}')


In [ ]:
# ── Cell 7: IFC relations tracked by the pipeline ─────────────────────────────
from pipeline.layer1_retriever.ifc_graph_builder import IFC_RELATION_CATEGORIES

print('IFC relations tracked by IFC-GraphRAG-DT:')
print(f'{"Relation Type":<48} {"Category":<25} Instances')
print('-' * 88)
for rel, cat in IFC_RELATION_CATEGORIES.items():
    count = len(model.by_type(rel))
    print(f'{rel:<48} {cat:<25} {count}')


In [ ]:
# ── Cell 8: KCS-DT smoke test ─────────────────────────────────────────────────
scorer = KCSDTScorer()
gt = {
    'entities':    [{'ifc_type': 'IfcPump'}, {'ifc_type': 'IfcPipeSegment'}, {'ifc_type': 'IfcValve'}],
    'relations':   [{'type': 'IfcRelConnectsPortToElement', 'from': 'IfcPump.OutletPort', 'to': 'IfcPipeSegment'},
                    {'type': 'IfcRelConnects', 'from': 'IfcPipeSegment', 'to': 'IfcValve'}],
    'attributes':  {'pump_0': {'material': 'cast_iron'}},
    'containment': [{'entity': 'IfcPump', 'container': 'IfcSpace'}],
    'connectivity':[{'from': 'IfcPump.OutletPort', 'to': 'IfcPipeSegment'}]
}
perfect = scorer.score(gt, gt, prompt_id='smoke-test')
print(f'KCS-DT perfect score: {perfect}')
assert abs(perfect.total - 1.0) < 1e-4, f'Expected 1.0, got {perfect.total}'

empty_pred = {'entities':[],'relations':[],'attributes':{},'containment':[],'connectivity':[]}
empty = scorer.score(empty_pred, gt)
print(f'KCS-DT empty pred:    {empty}')
assert empty.entity == 0.0
print('KCS-DT smoke test PASSED.')


In [ ]:
# ── Cell 9: GGS smoke test ────────────────────────────────────────────────────
ggs_scorer = GGSScorer()

G = nx.DiGraph()
G.add_node('A', ifc_type='IfcPump',        name='P-01')
G.add_node('B', ifc_type='IfcPipeSegment', name='Pipe-01')
G.add_node('C', ifc_type='IfcValve',       name='V-01')
G.add_edge('A', 'B', relation_type='IfcRelConnectsPortToElement')
G.add_edge('B', 'C', relation_type='IfcRelConnects')

ggs = ggs_scorer.score(G, G, prompt_id='smoke-test')
print(f'GGS perfect score: {ggs}')
assert abs(ggs.node_recall - 1.0) < 1e-4
assert abs(ggs.edge_recall - 1.0) < 1e-4
print('GGS smoke test PASSED.')
print('\n' + '='*55)
print('All checks passed. Ready for Notebook 01.')
print('='*55)
